In [36]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib
import os

TITLES = ["Train_1", "Train_2", "Test_1", "Test_2", "Test_3", "Val", "LSG_1", "LSG_2"]
PREDICTORS = ["PwmD", "PwmE", "sPwm", "dPwm"]   
TARGET_INT = ["Theta"]  
TARGET = ["DeltaTheta"]
     
INPUT_SIZE = len(PREDICTORS)  
OUTPUT_SIZE = len(TARGET)   
     
TIME_STEPS = 3
TS = 0.07

In [37]:
Datasets = []
for i in range(4):
    Dataset = pd.read_excel(f"../../00-RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"../../00-Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS
    
    Dataset["sPwm"] = Dataset["PwmD"] + Dataset["PwmE"]
    Dataset["dPwm"] = Dataset["PwmD"] - Dataset["PwmE"]

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [38]:

NormDatasets = []

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

Train1 = Datasets[0].copy()
Train1[PREDICTORS] = SCALER.fit_transform(Train1[PREDICTORS])
Train1[TARGET] = OUT_SCALER.fit_transform(Train1[TARGET])
NormDatasets.append(Train1)

Train2 = Datasets[1].copy()
Train2[PREDICTORS] = SCALER.transform(Train2[PREDICTORS])
Train2[TARGET] = OUT_SCALER.transform(Train2[TARGET])
NormDatasets.append(Train2)

# concatena
Train = pd.concat([Train1, Train2], ignore_index=True)

for i in range(6):
    CurrentTestDataset = Datasets[i + 2].copy()
    CurrentTestDataset[PREDICTORS] = SCALER.transform(CurrentTestDataset[PREDICTORS])
    CurrentTestDataset[TARGET] = OUT_SCALER.transform(CurrentTestDataset[TARGET])
    NormDatasets.append(CurrentTestDataset)

In [39]:
os.makedirs("./scalers", exist_ok=True)
os.makedirs("./Data", exist_ok=True)

with pd.ExcelWriter("./Data/NormDatasets.xlsx", engine="openpyxl") as writer_norm:
    for title, normDataset in zip(TITLES, NormDatasets):
        normDataset.to_excel(writer_norm, sheet_name=title[:31], index=False)

with pd.ExcelWriter("./Data/Datasets.xlsx", engine="openpyxl") as writer:
    for title, Dataset in zip(TITLES, Datasets):
        Dataset.to_excel(writer, sheet_name=title[:31], index=False)
        
joblib.dump(SCALER, "./scalers/scaler.pkl")
joblib.dump(OUT_SCALER, "./scalers/out_scaler.pkl")

mean_tf = tf.constant(OUT_SCALER.mean_[0], dtype=tf.float32)
std_tf  = tf.constant(OUT_SCALER.scale_[0], dtype=tf.float32)        

In [40]:
def CreateSequences(input_data, target_data, timesteps):
    X_seq, Y_seq = [], []
    
    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i])
    return np.array(X_seq), np.array(Y_seq)

x_train, y_train = CreateSequences(Train[PREDICTORS], Train[TARGET], TIME_STEPS)

x_val, y_val = CreateSequences((NormDatasets[5])[PREDICTORS], (NormDatasets[5])[TARGET], TIME_STEPS)
print(f"Dimensão da entrada: {np.shape(x_train)}")
print(f"Dimensão da saida: {np.shape(y_train)}")

print(f"Dimensão da entrada: {np.shape(x_val)}")
print(f"Dimensão da saida: {np.shape(y_val)}")

Dimensão da entrada: (1949, 3, 4)
Dimensão da saida: (1949, 1)
Dimensão da entrada: (1268, 3, 4)
Dimensão da saida: (1268, 1)


In [41]:
# =========================
# 1. Determinismo
# =========================
def MakeDetrministic(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.config.experimental.enable_op_determinism()

MakeDetrministic(42)

In [42]:
# =========================
# 3. Modelo
# =========================
initializer = initializers.RandomNormal(seed=int(42))
regularizer = tf.keras.regularizers.l2(0.9)

model = tf.keras.Sequential()
model.add(tf.keras.layers.Input(shape=(TIME_STEPS, INPUT_SIZE)))

model.add(tf.keras.layers.SimpleRNN(
                        10,
                        activation='tanh',
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

model.add(tf.keras.layers.Dense(
            OUTPUT_SIZE,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        ))


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='mse'
)

In [53]:

# =========================
# 4. Treinamento
# =========================
model.fit(x_train, y_train, epochs=50, batch_size=10, verbose=0, shuffle=False)

In [54]:
# =========================
# 5. Salvar modelo
# =========================
model.save("rnn_model.keras")

In [55]:
# =========================
# 6. Feedforward 
# =========================

y_pred_o = model.predict(x_val[:5])
print("Predição modelo original:", y_pred_o.flatten())

1/1 [==============================] - 0s 62ms/step
Predição modelo original: [0.07491776 0.07491778 0.07484753 0.07484754 0.07477798]


In [56]:
[0.07520892 0.07520892 0.07520892 0.07520892 0.07520892]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2465818368.py, line 1)